# Industrial CNC Machine Monitoring

## Exploratory Data Analysis of Operational Variables and Sensor Measurements

### 1. Introduction

This project focuses on the exploratory analysis of industrial CNC machine data.

The objective is to understand the machine's operational behavior by combining two data sources: CNC operational variables and external sensor measurements.

### 2. Dataset Description

The analysis uses two Excel datasets:

- **CNCVariables:** operational variables, including axis loads, speeds, positions, power, intensity and temperatures.
- **SensorData:** measurements from 14 Bar channels and 49 thermocouples.

The datasets contain approximately 179,048 records combined and 106 columns.

### 3. Business Objective

Understand the operational and thermal behavior of the CNC machine, identify statistical patterns and explore potential relationships between machine operation and sensor measurements.

### 4. Research Questions

- RQ01: How does the CNC machine behave during the observation period?
- RQ02: How do temperature measurements differ across thermocouples?
- RQ03: Are machine loads associated with temperature variations?
- RQ04: Is spindle speed associated with thermal behavior?
- RQ05: Are there identifiable heating and cooling patterns?
- RQ06: Can statistically unusual operating conditions be identified?

### 5. Methodology

1. Data Understanding
2. Data Cleaning
3. Univariate Analysis
4. Bivariate and Multivariate Analysis
5. Feature Engineering
6. Time Series Analysis
7. Anomaly Exploration
8. Conclusions

### 6. Limitations

- No confirmed failure labels are available.
- Physical sensor locations are not documented.
- Bar channel meanings and measurement units require verification.
- Correlation does not imply causation.
- Predictive modeling is outside the initial EDA scope.


In [1]:

# Data manipulation
import pandas as pd
import numpy as np

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display configuration
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.3f}".format)

sns.set_theme(style="whitegrid")

print("Libraries imported successfully")


Libraries imported successfully


In [2]:

# Define file paths
cnc_path = "https://zenodo.org/records/16579804/files/CNCVariables6to8November2024.xlsx?download=1"
sensors_path = "https://zenodo.org/records/16579804/files/SensorData6To8November2024.xlsx?download=1"

# Load datasets
cnc = pd.read_excel(cnc_path)
sensors = pd.read_excel(sensors_path)

print("Datasets loaded successfully")


Datasets loaded successfully


In [3]:

print("CNC Dataset")
print(f"Rows: {cnc.shape[0]:,}")
print(f"Columns: {cnc.shape[1]:,}")

print("\nSensor Dataset")
print(f"Rows: {sensors.shape[0]:,}")
print(f"Columns: {sensors.shape[1]:,}")

print("\nTotal records:", f"{len(cnc) + len(sensors):,}")


CNC Dataset
Rows: 89,655
Columns: 41

Sensor Dataset
Rows: 89,393
Columns: 65

Total records: 179,048


In [ ]:
display(cnc.head())

display(sensors.head())

,Date,Timestamp,AAxisIntensity,AAxisLoad,AAxisPositionActual,AAxisPower,AAxisSpeed,CAxisIntensity,CAxisLoad,CAxisPositionActual,CAxisPower,CAxisSpeed,CAxisTemperature,ControllerMode,ExecutionState,PathFeedrateActual,PathFeedrateOverride,SPActualSpeed,SPAxisIntensity,SPAxisLoad,SPAxisPower,SPAxisTemperature,SpindleOverride,XAxisIntensity,XAxisLoad,XAxisPositionActual,XAxisPower,XAxisSpeed,XAxisTemperature,YAxisIntensity,YAxisLoad,YAxisPositionActual,YAxisPower,YAxisSpeed,YAxisTemperature,ZAxisIntensity,ZAxisLoad,ZAxisPositionActual,ZAxisPower,ZAxisSpeed,ZAxisTemperature
0,2024-11-06 06:47:47.362,1730875667.362,0.004,0.000,0.000,-0.000,0.000,0.018,0.000,149.134,0.000,0.000,22.000,0,5,0.000,0.000,0.000,0.000,0.000,0.000,24.052,100,0.000,0.000,-0.000,-0.000,0.000,20.500,0.000,0.000,-375.084,-0.000,0.000,19.300,0.000,0.000,375.084,-0.000,0.000,19.580
1,2024-11-06 06:47:48.362,1730875668.362,0.004,0.000,0.000,-0.000,0.000,0.018,0.000,149.134,0.000,0.000,22.000,0,5,0.000,0.000,0.000,0.000,0.000,0.000,24.052,100,0.000,0.000,-0.000,-0.000,0.000,20.500,0.000,0.000,-375.084,-0.000,0.000,19.300,0.000,0.000,375.084,-0.000,0.000,19.580
2,2024-11-06 06:47:49.363,1730875669.363,0.004,0.000,0.000,-0.000,0.000,0.018,0.000,149.134,-0.000,0.000,22.000,0,5,0.000,0.000,0.000,0.000,0.000,0.000,24.052,100,0.000,0.000,-0.000,0.000,0.000,20.500,0.000,0.000,-375.084,-0.000,0.000,19.300,0.000,0.000,375.084,0.000,0.000,19.558
3,2024-11-06 06:47:50.362,1730875670.362,0.004,0.000,0.000,0.000,0.000,0.018,0.000,149.134,0.000,0.000,22.000,0,5,0.000,0.000,0.000,0.000,0.000,-0.000,24.064,100,0.000,0.000,-0.000,0.000,0.000,20.500,0.000,0.000,-375.084,-0.000,0.000,19.300,0.000,0.000,375.084,0.000,0.000,19.558
4,2024-11-06 06:47:51.362,1730875671.362,0.004,0.000,0.000,-0.000,0.000,0.018,0.000,149.134,0.000,0.000,22.000,0,5,0.000,0.000,0.000,0.000,0.000,-0.000,24.026,100,0.000,0.000,-0.000,0.000,0.000,20.500,0.000,0.000,-375.084,-0.000,0.000,19.300,0.000,0.000,375.084,0.000,0.000,19.599


,Date,Timestamp,Bar_01,Bar_02,Bar_03,Bar_04,Bar_05,Bar_06,Bar_07,Bar_08,Bar_09,Bar_10,Bar_11,Bar_12,Bar_13,Bar_14,Thermocouple_01,Thermocouple_02,Thermocouple_03,Thermocouple_04,Thermocouple_05,Thermocouple_06,Thermocouple_07,Thermocouple_08,Thermocouple_09,Thermocouple_10,Thermocouple_11,Thermocouple_12,Thermocouple_13,Thermocouple_14,Thermocouple_15,Thermocouple_16,Thermocouple_17,Thermocouple_18,Thermocouple_19,Thermocouple_20,Thermocouple_21,Thermocouple_22,Thermocouple_23,Thermocouple_24,Thermocouple_25,Thermocouple_26,Thermocouple_27,Thermocouple_28,Thermocouple_29,Thermocouple_30,Thermocouple_31,Thermocouple_32,Thermocouple_33,Thermocouple_34,Thermocouple_35,Thermocouple_36,Thermocouple_37,Thermocouple_38,Thermocouple_39,Thermocouple_40,Thermocouple_41,Thermocouple_42,Thermocouple_43,Thermocouple_44,Thermocouple_45,Thermocouple_46,Thermocouple_47,Thermocouple_48,Thermocouple_49
0,2024-11-06 06:50:07,1730875807,58962.570,61182.460,60312.920,58430.210,63274.950,61873.820,58747.520,60691.690,59931.140,61356.850,58127.300,59703.950,60970.130,58614.510,18.700,17.800,16.900,17.800,17.900,19.100,18.100,16.100,18.600,17.400,16.900,17.800,17.800,18.600,17.800,16.100,18.600,17.700,16.900,18.100,17.800,19.000,18.000,16.100,18.600,17.400,16.500,17.400,17.400,18.500,17.600,15.900,18.600,17.900,16.800,17.700,17.900,19.000,18.100,16.100,18.400,17.300,16.800,17.900,18.100,18.800,16.800,14.800,16.800
1,2024-11-06 06:50:08,1730875808,58962.590,61182.440,60312.950,58430.260,63274.950,61873.890,58747.470,60691.730,59931.170,61356.800,58127.260,59704.000,60970.170,58614.540,18.700,17.800,16.900,17.800,17.900,19.200,18.100,16.100,18.600,17.400,16.800,17.800,17.800,18.600,17.800,16.100,18.600,17.700,16.900,18.100,17.800,19.000,18.100,16.100,18.600,17.400,16.600,17.400,17.400,18.600,17.600,15.900,18.600,17.900,16.800,17.700,17.900,19.000,18.100,16.100,18.400,17.300,16.800,17.900,18.100,18.800,16.700,14.800,16.800
2,2024-11-06 06:50:09,1730875809,58962.620,61182.460,60312.950,58430.260,63275.020,61873.840,58747.500,60691.730,59931.140,61356.800,58127.260,59703.950,60970.170,58614.560,18.700,17.800,16.900,17.800,17.900,19.100,18.100,16.100,18.600,17.400,16.900,17.800,17.800,18.600,17.800,16.100,18.600,17.700,16.900,18.100,17.800,19.000,18.100,16.100,18.600,17.400,16.600,17.400,17.400,18.500,17.600,15.900,18.600,17.900,16.800,17.700,17.900,19.000,18.100,16.100,18.400,17.300,16.800,17.900,18.100,18.800,16.700,14.800,16.800
3,2024-11-06 06:50:10,1730875810,58962.570,61182.460,60312.920,58430.230,63274.950,61873.890,58747.500,60691.690,59931.120,61356.850,58127.300,59703.950,60970.170,58614.510,18.700,17.800,16.900,17.900,17.900,19.100,18.100,16.100,18.600,17.400,16.800,17.800,17.800,18.600,17.800,16.100,18.600,17.800,16.900,18.100,17.800,19.000,18.100,16.200,18.600,17.400,16.600,17.400,17.400,18.600,17.600,15.900,18.600,17.900,16.800,17.700,17.900,19.000,18.100,16.100,18.400,17.300,16.800,17.900,18.100,18.800,16.800,14.800,16.800
4,2024-11-06 06:50:11,1730875811,58962.640,61182.510,60312.920,58430.210,63274.980,61873.840,58747.450,60691.730,59931.190,61356.820,58127.300,59704.000,60970.150,58614.510,18.700,17.800,16.900,17.800,17.900,19.200,18.100,16.100,18.600,17.400,16.900,17.800,17.800,18.600,17.800,16.100,18.600,17.800,16.900,18.100,17.800,19.000,18.000,16.200,18.600,17.400,16.600,17.400,17.400,18.600,17.600,15.900,18.600,17.900,16.800,17.700,17.900,19.000,18.100,16.100,18.400,17.300,16.800,17.900,18.100,18.900,16.700,14.800,16.700


In [6]:

print("CNC DATASET INFORMATION")
cnc.info()

print("\nSENSOR DATASET INFORMATION")
sensors.info()


CNC DATASET INFORMATION
<class 'pandas.DataFrame'>
RangeIndex: 89655 entries, 0 to 89654
Data columns (total 41 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   Date                  89655 non-null  datetime64[us]
 1   Timestamp             89655 non-null  float64       
 2   AAxisIntensity        89655 non-null  float64       
 3   AAxisLoad             89655 non-null  float64       
 4   AAxisPositionActual   89655 non-null  float64       
 5   AAxisPower            89655 non-null  float64       
 6   AAxisSpeed            89655 non-null  float64       
 7   CAxisIntensity        89655 non-null  float64       
 8   CAxisLoad             89655 non-null  float64       
 9   CAxisPositionActual   89655 non-null  float64       
 10  CAxisPower            89655 non-null  float64       
 11  CAxisSpeed            89655 non-null  float64       
 12  CAxisTemperature      89655 non-null  float64       
 13  Con

In [7]:

# CNC variables
cnc_numeric = cnc.select_dtypes(include=np.number).columns.tolist()
cnc_categorical = cnc.select_dtypes(include=["object", "category"]).columns.tolist()
cnc_datetime = cnc.select_dtypes(include=["datetime"]).columns.tolist()

# Sensor variables
sensor_numeric = sensors.select_dtypes(include=np.number).columns.tolist()
sensor_categorical = sensors.select_dtypes(include=["object", "category"]).columns.tolist()
sensor_datetime = sensors.select_dtypes(include=["datetime"]).columns.tolist()

print("CNC")
print("Numeric:", len(cnc_numeric))
print("Categorical:", len(cnc_categorical))
print("Datetime:", len(cnc_datetime))

print("\nSENSORS")
print("Numeric:", len(sensor_numeric))
print("Categorical:", len(sensor_categorical))
print("Datetime:", len(sensor_datetime))


CNC
Numeric: 40
Categorical: 0
Datetime: 1

SENSORS
Numeric: 64
Categorical: 0
Datetime: 1


In [8]:

display(cnc.describe().T)

display(sensors.describe().T)


,count,mean,min,25%,50%,75%,max,std
Date,89655,2024-11-07 08:49:28.229952,2024-11-06 06:47:47.362000,2024-11-06 13:01:21.971000,2024-11-07 10:08:40.110000,2024-11-08 07:28:23.743000,2024-11-08 13:41:58.388000,NaN
Timestamp,89655.000,1730969368.230,1730875667.362,1730898081.971,1730974120.110,1731050903.743,1731073318.388,67668.114
AAxisIntensity,89655.000,0.066,0.000,0.000,0.000,0.000,35.998,0.797
AAxisLoad,89655.000,0.180,0.000,0.000,0.000,0.000,43.758,2.126
AAxisPositionActual,89655.000,-51.345,-135.000,-90.000,-90.000,0.000,30.000,45.029
AAxisPower,89655.000,0.000,-0.124,-0.000,0.000,0.000,0.960,0.010
AAxisSpeed,89655.000,0.175,-7200.000,0.000,0.000,0.000,5000.000,87.835
CAxisIntensity,89655.000,0.584,0.000,0.000,0.272,1.293,50.302,0.673
CAxisLoad,89655.000,0.207,0.000,0.000,0.095,0.460,9.130,0.224
CAxisPositionActual,89655.000,154.408,0.000,45.000,149.584,225.000,360.000,120.545


,count,mean,min,25%,50%,75%,max,std
Date,89393,2024-11-07 08:49:57.705122,2024-11-06 06:50:07,2024-11-06 13:02:37,2024-11-07 10:09:44,2024-11-08 07:30:17,2024-11-08 13:42:47,NaN
Timestamp,89393.000,1730969397.705,1730875807.000,1730898157.000,1730974184.000,1731051017.000,1731073367.000,67641.085
Bar_01,89393.000,58904.206,58820.200,58876.480,58895.660,58936.210,58967.590,37.532
Bar_02,89393.000,61143.249,61091.540,61122.390,61142.460,61166.840,61186.390,27.039
Bar_03,89393.000,60284.007,60251.140,60268.810,60279.730,60302.090,60314.070,19.100
Bar_04,89393.000,58399.072,58359.460,58384.510,58394.980,58415.030,58433.040,19.914
Bar_05,89393.000,63205.964,63110.070,63140.570,63209.980,63275.210,63281.630,67.082
Bar_06,89393.000,61876.094,61862.340,61867.240,61876.320,61884.210,61892.010,9.010
Bar_07,89393.000,58737.599,58723.050,58731.630,58737.210,58743.540,58749.220,7.127
Bar_08,89393.000,60671.224,60650.430,60661.440,60667.770,60681.290,60692.010,11.746


In [9]:

for name, df in [("CNC", cnc), ("Sensors", sensors)]:

    print(f"\n{name} DATASET")

    print("Start:", df["Date"].min())
    print("End:", df["Date"].max())

    print("Duration:")
    print(df["Date"].max() - df["Date"].min())



CNC DATASET
Start: 2024-11-06 06:47:47.362000
End: 2024-11-08 13:41:58.388000
Duration:
2 days 06:54:11.026000

Sensors DATASET
Start: 2024-11-06 06:50:07
End: 2024-11-08 13:42:47
Duration:
2 days 06:52:40


In [10]:

# Check missing values

print("CNC Dataset:")
print("Missing values:", cnc.isnull().sum().sum())

print("\nSensor Dataset:")
print("Missing values:", sensors.isnull().sum().sum())


CNC Dataset:
Missing values: 0

Sensor Dataset:
Missing values: 0


In [11]:
# Check duplicated rows

print("CNC Dataset:")
print("Duplicated rows:", cnc.duplicated().sum())

print("\nSensor Dataset:")
print("Duplicated rows:", sensors.duplicated().sum())

CNC Dataset:
Duplicated rows: 0

Sensor Dataset:
Duplicated rows: 0


In [12]:
# Check sampling frequency

cnc["Date"] = pd.to_datetime(cnc["Date"])
sensors["Date"] = pd.to_datetime(sensors["Date"])

print("CNC sampling intervals:")
print(cnc["Date"].diff().value_counts().head())

print("\nSensor sampling intervals:")
print(sensors["Date"].diff().value_counts().head())

CNC sampling intervals:
Date
0 days 00:00:01           78571
0 days 00:00:01.001000     5201
0 days 00:00:00.999000     5184
0 days 00:00:00.998000       87
0 days 00:00:01.002000       80
Name: count, dtype: int64

Sensor sampling intervals:
Date
0 days 00:00:01    89374
0 days 00:00:02       12
0 days 00:00:00        4
0 days 14:54:38        1
0 days 15:08:04        1
Name: count, dtype: int64


In [13]:
# Identify large gaps in sensor timestamps

sensor_diff = sensors["Date"].diff()

large_gaps = sensors.loc[
    sensor_diff > pd.Timedelta(minutes=5),
    ["Date"]
].copy()

large_gaps["Previous_Date"] = sensors["Date"].shift(1)
large_gaps["Gap"] = sensor_diff[sensor_diff > pd.Timedelta(minutes=5)]

large_gaps

,Date,Previous_Date,Gap
32606,2024-11-07 06:48:13,2024-11-06 15:53:35,0 days 14:54:38
65123,2024-11-08 06:58:16,2024-11-07 15:50:12,0 days 15:08:04


In [14]:
# Identify duplicated sensor timestamps

duplicated_dates = sensors[
    sensors["Date"].duplicated(keep=False)
].sort_values("Date")

duplicated_dates[["Date", "Timestamp"]]

,Date,Timestamp
81466,2024-11-08 11:30:41,1731065441
81467,2024-11-08 11:30:41,1731065441
81475,2024-11-08 11:30:50,1731065450
81476,2024-11-08 11:30:50,1731065450
81549,2024-11-08 11:32:04,1731065524
81550,2024-11-08 11:32:04,1731065524
81555,2024-11-08 11:32:10,1731065530
81556,2024-11-08 11:32:10,1731065530


In [15]:
# Compare sensor measurements for duplicated timestamps

duplicate_rows = sensors[
    sensors["Date"].duplicated(keep=False)
].sort_values("Date")

duplicate_rows.T

,81466,81467,81475,81476,81549,81550,81555,81556
Date,2024-11-08 11:30:41,2024-11-08 11:30:41,2024-11-08 11:30:50,2024-11-08 11:30:50,2024-11-08 11:32:04,2024-11-08 11:32:04,2024-11-08 11:32:10,2024-11-08 11:32:10
Timestamp,1731065441,1731065441,1731065450,1731065450,1731065524,1731065524,1731065530,1731065530
Bar_01,58880.320,58880.300,58880.280,58880.300,58880.180,58880.180,58880.210,58880.210
Bar_02,61128.550,61128.570,61128.620,61128.550,61128.430,61128.410,61128.410,61128.430
Bar_03,60269.130,60269.110,60269.110,60269.090,60269.060,60269.090,60269.090,60269.060
Bar_04,58385.270,58385.290,58385.250,58385.250,58385.220,58385.220,58385.250,58385.220
Bar_05,63115.890,63115.890,63115.910,63115.840,63116.090,63116.050,63116.120,63116.090
Bar_06,61866.370,61866.340,61866.320,61866.370,61866.250,61866.300,61866.300,61866.270
Bar_07,58731.630,58731.740,58731.630,58731.650,58731.610,58731.610,58731.630,58731.610
Bar_08,60664.250,60664.250,60664.250,60664.250,60664.200,60664.270,60664.270,60664.250


In [16]:
# Check CNC sampling intervals

cnc_diff = cnc["Date"].diff()

print("CNC sampling intervals:")
print(cnc_diff.value_counts().head(10))

print("\nIntervals longer than 5 minutes:")
print(cnc_diff[cnc_diff > pd.Timedelta(minutes=5)])

CNC sampling intervals:
Date
0 days 00:00:01           78571
0 days 00:00:01.001000     5201
0 days 00:00:00.999000     5184
0 days 00:00:00.998000       87
0 days 00:00:01.002000       80
0 days 00:00:01.003000       62
0 days 00:00:00.997000       57
0 days 00:00:00.996000       38
0 days 00:00:01.004000       35
0 days 00:00:01.017000       33
Name: count, dtype: int64

Intervals longer than 5 minutes:
32679   0 days 14:53:44.543000
65260   0 days 15:06:10.017000
Name: Date, dtype: timedelta64[us]


In [17]:
# Identify large gaps in CNC timestamps

large_gaps_cnc = cnc.loc[
    cnc_diff > pd.Timedelta(minutes=5),
    ["Date"]
].copy()

large_gaps_cnc["Previous_Date"] = cnc["Date"].shift(1)
large_gaps_cnc["Gap"] = cnc_diff[
    cnc_diff > pd.Timedelta(minutes=5)
]

large_gaps_cnc

,Date,Previous_Date,Gap
32679,2024-11-07 06:46:11.529,2024-11-06 15:52:26.986,0 days 14:53:44.543000
65260,2024-11-08 06:55:23.173,2024-11-07 15:49:13.156,0 days 15:06:10.017000


In [18]:
# Identify duplicated CNC timestamps

duplicated_cnc_dates = cnc[
    cnc["Date"].duplicated(keep=False)
].sort_values("Date")

print("Duplicated timestamps:", cnc["Date"].duplicated().sum())

duplicated_cnc_dates[["Date", "Timestamp"]]

Duplicated timestamps: 0


,Date,Timestamp


In [19]:
# Check chronological order

print("CNC chronologically sorted:", cnc["Date"].is_monotonic_increasing)
print("Sensors chronologically sorted:", sensors["Date"].is_monotonic_increasing)

CNC chronologically sorted: True
Sensors chronologically sorted: True


In [20]:
# Compare timestamp data types

print("CNC:")
print(cnc[["Date", "Timestamp"]].dtypes)

print("\nSensors:")
print(sensors[["Date", "Timestamp"]].dtypes)

CNC:
Date         datetime64[us]
Timestamp           float64
dtype: object

Sensors:
Date         datetime64[us]
Timestamp             int64
dtype: object


In [21]:
# Inspect CNC timestamp precision

print("CNC timestamp examples:")
print(cnc["Timestamp"].head(10).to_list())

print("\nSensor timestamp examples:")
print(sensors["Timestamp"].head(10).to_list())

CNC timestamp examples:
[1730875667.3620002, 1730875668.3619998, 1730875669.3630002, 1730875670.3620002, 1730875671.362, 1730875672.3630004, 1730875673.363, 1730875674.3630002, 1730875675.3629997, 1730875676.363]

Sensor timestamp examples:
[1730875807, 1730875808, 1730875809, 1730875810, 1730875811, 1730875812, 1730875813, 1730875814, 1730875815, 1730875816]


In [22]:
# Compare timestamps at second resolution

cnc["Timestamp_Second"] = np.floor(cnc["Timestamp"]).astype("int64")

print("CNC unique seconds:", cnc["Timestamp_Second"].nunique())
print("Sensors unique seconds:", sensors["Timestamp"].nunique())

common_seconds = set(cnc["Timestamp_Second"]) & set(sensors["Timestamp"])

print("Common seconds:", len(common_seconds))
print("CNC seconds without sensor data:",
      len(set(cnc["Timestamp_Second"]) - set(sensors["Timestamp"])))
print("Sensor seconds without CNC data:",
      len(set(sensors["Timestamp"]) - set(cnc["Timestamp_Second"])))

CNC unique seconds: 89653
Sensors unique seconds: 89389
Common seconds: 89207
CNC seconds without sensor data: 446
Sensor seconds without CNC data: 182


In [23]:
# Count sensor records per timestamp second

sensor_counts = sensors["Timestamp"].value_counts()

print("Maximum records per second:", sensor_counts.max())

print("\nNumber of timestamps with multiple records:")
print((sensor_counts > 1).sum())

print("\nRecords per timestamp distribution:")
print(sensor_counts.value_counts().sort_index())

Maximum records per second: 2

Number of timestamps with multiple records:
4

Records per timestamp distribution:
count
1    89385
2        4
Name: count, dtype: int64


In [24]:
# Inspect duplicated sensor timestamps and row positions

duplicate_indices = sensors[
    sensors["Timestamp"].duplicated(keep=False)
].index

print("Duplicated row indices:")
print(duplicate_indices.tolist())

print("\nDuplicated timestamp groups:")
print(
    sensors.loc[duplicate_indices, ["Date", "Timestamp"]]
    .sort_values(["Timestamp", "Date"])
)

Duplicated row indices:
[81466, 81467, 81475, 81476, 81549, 81550, 81555, 81556]

Duplicated timestamp groups:
                     Date   Timestamp
81466 2024-11-08 11:30:41  1731065441
81467 2024-11-08 11:30:41  1731065441
81475 2024-11-08 11:30:50  1731065450
81476 2024-11-08 11:30:50  1731065450
81549 2024-11-08 11:32:04  1731065524
81550 2024-11-08 11:32:04  1731065524
81555 2024-11-08 11:32:10  1731065530
81556 2024-11-08 11:32:10  1731065530
